# TrustLung AI — Exploratory Data Analysis

This notebook explores:
- CT scan class distribution and sample images
- Clinical dataset feature distributions
- Correlation analysis
- Class imbalance visualization
- Data quality checks

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.helpers import load_config, set_seed
from src.preprocessing.data_loader import generate_synthetic_dataset, ClinicalDataPreprocessor

set_seed(42)
config = load_config('../configs/config.yaml')
print('Config loaded:', config['project']['name'])

## 1. Dataset Overview

In [ ]:
# Generate or load dataset
data = generate_synthetic_dataset(n_samples=600, seed=42)

print('Dataset summary:')
print(f'  Train: {len(data["X_train"])} samples')
print(f'  Val:   {len(data["X_val"])} samples')
print(f'  Test:  {len(data["X_test"])} samples')
print(f'  Image shape: {data["X_train"][0].shape}')
print(f'  Classes: {data["class_names"]}')
print(f'  Class weights: {data["class_weights"]}')

## 2. Class Distribution

In [ ]:
from src.visualization.plots import plot_class_distribution

plot_class_distribution(
    data['y_train_raw'],
    class_names=data['class_names'],
    title='Training Set Class Distribution',
)
plt.show()

## 3. Sample CT Scan Images

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.patch.set_facecolor('#0f172a')

for class_idx, class_name in enumerate(data['class_names']):
    mask    = data['y_train_raw'] == class_idx
    samples = data['X_train'][mask][:5]
    for col, img in enumerate(samples):
        ax = axes[class_idx, col]
        ax.imshow(img, cmap='gray')
        ax.set_title(class_name if col == 0 else '', color='white', fontsize=10)
        ax.axis('off')

plt.suptitle('Sample Images per Class', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Clinical Data EDA

In [ ]:
preprocessor = ClinicalDataPreprocessor()
clinical_df  = preprocessor.generate_demo_clinical_data(n_samples=300, seed=42)

print('Clinical data shape:', clinical_df.shape)
print('\nTarget distribution:')
print(clinical_df['LUNG_CANCER'].value_counts())
print('\nFeature types:')
print(clinical_df.dtypes)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('#0f172a')

binary_feats = ['SMOKING', 'COUGHING', 'CHEST_PAIN', 'FATIGUE', 'WHEEZING']

# Age distribution
ax = axes[0, 0]
ax.set_facecolor('#1e293b')
for label, color in [('YES', '#ef4444'), ('NO', '#4ade80')]:
    subset = clinical_df[clinical_df['LUNG_CANCER'] == label]['AGE']
    ax.hist(subset, bins=20, alpha=0.7, label=label, color=color)
ax.set_title('Age Distribution by Cancer Status', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#475569')
ax.legend(facecolor='#1e293b', labelcolor='white')

# Smoking
ax = axes[0, 1]
ax.set_facecolor('#1e293b')
ct = pd.crosstab(clinical_df['SMOKING'], clinical_df['LUNG_CANCER'])
ct.plot(kind='bar', ax=ax, color=['#4ade80', '#ef4444'], alpha=0.8, edgecolor='none')
ax.set_title('Smoking vs Cancer', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#475569')
ax.legend(facecolor='#1e293b', labelcolor='white')

# Gender
ax = axes[0, 2]
ax.set_facecolor('#1e293b')
ct2 = pd.crosstab(clinical_df['GENDER'], clinical_df['LUNG_CANCER'])
ct2.plot(kind='bar', ax=ax, color=['#4ade80', '#ef4444'], alpha=0.8, edgecolor='none')
ax.set_title('Gender vs Cancer', color='white')
ax.tick_params(colors='white')
ax.spines[:].set_color('#475569')

# Correlation heatmap
ax = axes[1, :2]
ax = plt.subplot2grid((2, 3), (1, 0), colspan=2)
ax.set_facecolor('#1e293b')
num_df = clinical_df.copy()
num_df['LUNG_CANCER'] = (num_df['LUNG_CANCER'] == 'YES').astype(int)
num_df['GENDER'] = (num_df['GENDER'] == 'M').astype(int)
corr = num_df.select_dtypes(include='number').corr()
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0,
            annot=True, fmt='.2f', linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', color='white')
ax.tick_params(colors='white')

plt.suptitle('Clinical Data EDA', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/plots/clinical_eda.png', bbox_inches='tight', facecolor='#0f172a', dpi=150)
plt.show()